In [19]:
# Find_Non_Denom_Name_Plots.py
# 8/15/2025

from bs4 import BeautifulSoup 
import datetime
from datetime import datetime as dt
from datetime import date 
from datetime import timedelta
import time
import re
from pandas import read_csv
import pandas as pd
import json

import sys
import random

import requests
import math
import numpy as np
import urllib3
import inspect
from geopy.distance import distance
# from IPython.display import display


# def maximize_screen():
#     from IPython.display import display, HTML
#     display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def left(s, amount):
    return s[:amount]
def right(s, amount):
    return s[-amount:]
def mid(s, offset, amount):
    return s[offset:offset+amount]

def maximize_screen():
    from IPython.display import display, HTML
    display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def timestamp():
    today=str(datetime.date.today()).split('-')
    year=today[0]
    month=today[1]
    day=today[2]
    a=str(datetime.datetime.now())
    time_split=re.split(':| ',a)
    hour=time_split[1]
    timestamp=year+month+day+hour
    return timestamp

def convert_date_z(date): 
    date_t= pd.to_datetime(date, utc=True)
    date_out=date_t.dt.strftime("%m/%d/%Y")
    return date_out

def convert_datetime_t(event_date_t):
    no_time='2000-01-01T00:00:00'
    no_time_dt=dt.strptime(no_time, "%Y-%m-%dT%H:%M:%S") 
    event_date_t = event_date_t.fillna(no_time)
    event_date_dt=pd.to_datetime(event_date_t, format="%Y-%m-%dT%H:%M:%S")
    event_date_index=event_date_dt.dt.strftime("%Y%m%d")
    event_date =event_date_dt.dt.strftime("%m/%d/%Y")
    event_date_rnd_dt = event_date_dt + timedelta(minutes=30)
    event_time_index=event_date_rnd_dt.dt.strftime('%H')  
    event_time = event_date_dt.dt.strftime('%H:%M')
    return     event_date,event_date_index,event_time,event_time_index 

def display_dataframe_segments(dataframe_df):
    for i in range(0,len(dataframe_df),50):
        print(dataframe_df.iloc[i:i+50])


def random_sleep(call_count): 
    global sleep_time_pause,sleep_range,pause_count
    if call_count%pause_count==0:
        sleep_time =sleep_time_pause
        now_date = datetime.datetime.now() + timedelta(seconds=sleep_time)
        sleep_end_date=now_date.strftime("%m/%d/%Y %H:%M:%S")    
        print(f"\t\t\tSleeping for {sleep_time:.0f} seconds • Sleeping till: {sleep_end_date}")
    else:
        sleep_time = random.uniform(sleep_range[0],sleep_range[1])
    return sleep_time 

def get_grave_count(url):
    global status_code
    capture_json=[]
    grave_count=0
    response = requests.request("GET", url, headers=my_header)
    status_code=response.status_code
    url_source = response.text       
    if status_code!=200:
        iTry=0
        while iTry <=4:
            print('\n** Requests Are Too Fast *** , Try=',iTry,'\n')
            time.sleep(60)
            iTry+=1
            response = requests.request("GET", url, headers=my_header)
            status_code=response.status_code
            url_source = response.text 
            if status_code==200:break
    beautifulSoupText = BeautifulSoup(url_source, 'html.parser')
    specific_tag_text = beautifulSoupText.get_text()
    capture_json=json.loads(specific_tag_text)
    try:
        grave_count=capture_json['total']
    except:
        specific_tag_text
        print(capture_json)
        print(f" XXXX get_grave_count Failed: Try: {iTry} \n url: {url}")
        time.sleep(60)
    return capture_json,grave_count
    
def extract_grave_data(capture_json):
    global field_list_set,field_list
    collection_df=pd.json_normalize(capture_json['collection'])
    collection_columns_set=set(list(collection_df))
    missing_columns=field_list_set-collection_columns_set
    collection_df[list(missing_columns)]=np.nan
    try:
        collection_df['links']=collection_df['links'].str[0].str[2]
    except:
        collection_df['links']=np.nan
    grave_frame_df=collection_df[field_list]
    return grave_frame_df

def clean_dataframe(graves_df):
    global field_list
    for column in field_list:
        graves_df[column]= graves_df[column].astype(str).replace(r"\,", ' * ', regex=True).str.strip()
        graves_df[column]= graves_df[column].astype(str).replace(r"\'", '', regex=True).str.strip()
    delete_list=[r'\"',r"\'",r'\?',r'\[',r'\]']
    for item in delete_list:
        for column in field_list:
            graves_df[column]= graves_df[column].astype(str).replace(item, ' ', regex=True).str.strip()
    return graves_df

# def trim_dataframe(dataframe_df,column):
#     drop_list=['AME','Anglican','antioch','Apos','Asylum','Baptist','Bishop','Calvary','Campo',\
#                'Canaan','Cathedral','Catholic','Cementerio','Cemit','Centenary','Chapel','Christ',\
#                'Church','Cimet','Cimiter','Cintor','CME','Cmentarz','Colored','County','Deutscher',\
#                'Dominican','Ebenezer','Friedhof','Gospel','Institute','Knight','Lutheran',\
#                'Masonic ','Mennonite ','Methodist','Military','Missionary','Parish ','Presbyterian',\
#                'prison','Protestant','public','Resurrection','Saint','St.','State','Township','Veterans']
#     for word in drop_list:
#         dataframe_df.drop(dataframe_df[dataframe_df[column].str.contains(word, case=False, na=False)].index, inplace=True) 
#     dataframe_df.reset_index(drop=True, inplace = True)   
#     return dataframe_df

def random_sleep_new(get_counter):
    global sleep_time_pause,sleep_range,pause_count
    get_counter+=1
    if get_counter%pause_count==0:
        sleep_time =sleep_time_pause
        now_date = datetime.datetime.now() + timedelta(seconds=sleep_time)
        sleep_end_date=now_date.strftime("%m/%d/%Y %H:%M:%S")    
        print(f"\t\t\tSleeping for {sleep_time:.0f} seconds   Sleeping till: {sleep_end_date}")
    else:
        sleep_time = random.uniform(sleep_range[0],sleep_range[1])
    time.sleep(sleep_time)    
    return get_counter

my_header = {
  'priority': 'u=1, i',
  'sec-ch-ua': '"Google Chrome";v="135", "Not-A.Brand";v="8", "Chromium";v="135"',
  'sec-ch-ua-platform': '"Windows"',
  'sec-fetch-mode': 'cors',
  'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36',
  'Cookie': '__cf_bm=O55WB_mmMt72i8KoQuzWliizNTb6J7zg2RVHXqbn.vc-1745620049-1.0.1.1-IDSlPNW4kRjBMAr5n8M2obOoGp9GAod_BHbB997Ze2F0YUm2DvAFh5TaijsQhPWR0iyIyFoaSy.p1YDke07ZttHHC83SJpQCEBFmaeImKIc; _cfuvid=zhvP.IczaAr9sd4YVjWjTqBayGxsei_Ntl0ANgSCCBg-1745620049033-0.0.1.1-604800000; ipLoc=us; preferredLanguage=en'
}

blank_line='                                                                                                                                                                                           '


#*************** Append  *******************************
append=0
# append_flag=input('To Append Press 1')
# if append_flag=='1':append=1
#*************** Append  *******************************

maximize_screen()
timestamp=timestamp()
collection_date0=datetime.datetime.now().strftime("%m/%d/%Y")
collection_date_dt=pd.to_datetime(collection_date0)
collection_date=collection_date0

start_time=datetime.datetime.now()
event_time=start_time

process_name='Find_Name Non Denom Plots'
print('**Starting *** ',process_name,' *** , Time:',start_time,' **\n')
DirMain="C:\\Users\\jrozm\\OneDrive\\Benjamin\\"


fileInput="ND_Name_List"
fileOutput="ND_Name_Graves_Plots"


fileIn=DirMain+"Affilation_Analysis\\Inputs\\"+fileInput+".csv"
fileOut=DirMain+"Affilation_Analysis\\Outputs\\"+fileOutput+".csv"
fileOutInt=DirMain+"Affilation_Analysis\\Outputs\\"+fileOutput+"_Int.csv"



nd_parents_names_df=pd.read_csv(fileIn)


field_list=['lastName','firstName','middleName','fullName','nickName','hasPlot','plot','birthDate','birthMonth','birthYear','deathDate','deathMonth',\
            'deathYear','birthCityName','birthStateName','birthCountryName','cemeteryName','countryName','cemeteryId','links', 'intermentHasPhoto','Parents','Spouses','Siblings','Children']
field_list_set=set(field_list)
if append==1:print('**** Appending ****')    
if append!=1:graves_df=pd.DataFrame([])  


url_first_name="https://www.findagrave.com/memorial/search?ajax=true&skip=&limit=100&firstname="
url_middle_name="&middlename="
url_last_name="&lastname="
url_suffix ="&exactName=true"


sleep_range=[5,12]
sleep_time_pause=1*60
pause_count=100
get_counter=0
cemetery_count=0
total_frame_count=0
name_count=0
total_name_count=len(nd_parents_names_df)
for index, row in nd_parents_names_df.iterrows():
    name_count+=1
    last_name=row['last_name']
    first_name=row['first_name']  
    middle_name=row['middle_name'] 
    frame_count=0
    grave_frame_count=0
    extracted_count=0
    grave_count=100
    print(f"\n *** {process_name:<20}  • Name: *{last_name}-{first_name}*  ••  Name Count: {name_count}/{total_name_count} ******* ")
    while grave_frame_count<grave_count:
        if pd.isna(middle_name)==True:middle_name=''
        url=url_first_name+first_name+url_middle_name+middle_name+url_last_name+last_name+url_suffix
        capture_json,grave_count=get_grave_count(url)
        get_counter=random_sleep_new(get_counter)
        if status_code!=200:print('\tstatus_code:',status_code);break 
        loop_count=0
        grave_frame_df=extract_grave_data(capture_json)                           
        grave_frame_df=clean_dataframe(grave_frame_df)        
        graves_df = pd.concat([graves_df,grave_frame_df])         
        grave_frame_count+=100
        extracted_count+=len(grave_frame_df)            
        graves_df.to_csv(fileOutInt,mode="w", index=False,header=True)
        
        print(f" {blank_line}", end='\r')
        print(f"    --- {process_name:<20}   • Name: *{last_name}-{first_name}* •   • grave count: {extracted_count}/{grave_count}", end='\r')
    print(f"*** {process_name:<20}   • Name: *{last_name}-{first_name}* •   • grave count: {extracted_count} • Name Count: {name_count}/{total_name_count}  ******* ")      
        
end_time=datetime.datetime.now()
processing_time=end_time-start_time
processing_time_seconds=int(processing_time.total_seconds())
print(f"**** {process_name}   Processing Time: {processing_time_seconds} Sec")   
graves_df=clean_dataframe(graves_df)  
graves_df.to_csv(fileOut,mode="w", index=False,header=True)
# graves_df.to_csv(fileOut0,mode="w", index=False,header=True) 
print("\n")
print('Uploaded To: ',fileOut)


    


**Starting ***  Find_Name Non Denom Plots  *** , Time: 2025-08-15 13:19:47.306575  **


 *** Find_Name Non Denom Plots  • Name: *Goodman-Anne*  ••  Name Count: 1/33 ******* 
*** Find_Name Non Denom Plots   • Name: *Goodman-Anne* •   • grave count: 0 • Name Count: 1/33  *******                                                                                     

 *** Find_Name Non Denom Plots  • Name: *Conn-Arthur*  ••  Name Count: 2/33 ******* 
*** Find_Name Non Denom Plots   • Name: *Conn-Arthur* •   • grave count: 2 • Name Count: 2/33  *******                                                                                      

 *** Find_Name Non Denom Plots  • Name: *Chase-Arthur*  ••  Name Count: 3/33 ******* 
*** Find_Name Non Denom Plots   • Name: *Chase-Arthur* •   • grave count: 1 • Name Count: 3/33  *******                                                                                     

 *** Find_Name Non Denom Plots  • Name: *Arth-Carl*  ••  Name Count: 4/33 ******* 
**

In [18]:

pd.isna(middle_name)

False